In [1]:
from dotenv import load_dotenv, find_dotenv

In [2]:
load_dotenv(find_dotenv())

True

In [3]:
from openai import OpenAI

In [4]:
client = OpenAI()

In [11]:
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)   

In [12]:
print(response.choices[0].message.content)

The capital of France is Paris.


In [14]:
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Suggest a name for a hotel room-service bot."},
    ],
    n=3
)   

In [15]:
for i, choice in enumerate(response.choices):
    print(i, choice.finish_reason, choice.message.content)
print(response.usage)

0 stop Here are 15 name ideas, grouped by tone, with a short tagline for each to show how it would read in use. Tell me which tone fits your hotel and I can refine names or craft a logo/voice brief.

Luxury
- Elysian Butler — “Discreet, five‑star room service.”
- Aurelia Concierge — “Bespoke service with a touch of gold.”

Friendly
- Roomie — “Your in‑room friend for everything you need.”
- Halo — “Always nearby, always helpful.”

Tech-forward
- InRoomIQ — “Smart service, instant answers.”
- BellBot — “Fast requests, flawless delivery.”
- GuestLink — “Connect. Request. Receive.”

Professional / Neutral
- ServiceOne — “One call for anything in your room.”
- RoomRequest — “Simple requests, quick fulfillment.”

Playful
- ButlerBee — “Busy helping you—buzz us anytime.”
- TrayMate — “Your trusty tray companion.”

Multilingual / Global
- Amico — “Friendly service” (Italian nuance)
- BonjourBot — “Bonjour, how can I help?” (French flair)
- Amigo — “Always at your service” (Spanish nuance)

If

In [5]:
from litellm import completion

In [17]:
response = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "Hello!"}]
)

## Listing 2.5: Statelessness

In [20]:
# First call
response1 = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is Sharon."}]
)

#Second call
response2 = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "What is my name?"}]
)

In [21]:
print(response2.choices[0].message.content)

I don't know your name — I don't have access to personal info unless you tell me. What would you like me to call you in this conversation? (If you want me to remember it across sessions, tell me if you have a memory/profile feature enabled.)


In [23]:
messages = []

#First exchange
messages.append({"role": "user", "content": "My name is Sharon!"})
response1 = completion(model="gpt-5-mini", messages=messages) 

In [24]:
assistant_message1 = response1.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_message1})

In [25]:
print(assistant_message1)

Nice to meet you, Sharon! How can I help you today?


In [26]:
#Second exchange - includes previous conversation history
messages.append({"role": "user", "content": "What is my name?"})
response2 = completion(model="gpt-5-mini", messages=messages)
assistant_message2 = response2.choices[0].message.content

In [27]:
print(assistant_message2)

Your name is Sharon.


In [28]:
print(response2.usage.prompt_tokens)

40


In [29]:
print(response1.usage.prompt_tokens)

11


In [6]:
from pydantic import BaseModel

In [7]:
class ExtractedInfo(BaseModel):
    name: str
    email: str
    phone: str | None = None

In [8]:
response = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is John Smith, my email is John@example.com, and my phone is 555-1234"}],
    response_format=ExtractedInfo
)

In [9]:
result = response.choices[0].message.content

In [10]:
print(type(result))

<class 'str'>


In [11]:
info = ExtractedInfo.model_validate_json(result)

In [12]:
info.email

'John@example.com'

In [13]:
info.phone

'555-1234'

In [14]:
response = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is John Smith, my email is John@example.com"}],
    response_format=ExtractedInfo
)

In [15]:
result = response.choices[0].message.content

In [16]:
info = ExtractedInfo.model_validate_json(result)

In [17]:
info.phone

In [18]:
print(result)

{"name":"John Smith","email":"John@example.com","phone":null}


In [19]:
response = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is John Smith"}],
    response_format=ExtractedInfo
)

In [20]:
result = response.choices[0].message.content

In [21]:
info = ExtractedInfo.model_validate_json(result)

In [22]:
info.email

''

In [23]:
print(type(info))

<class '__main__.ExtractedInfo'>


In [24]:
info.phone

In [25]:
ExtractedInfo.model_validate_json('{"name": "A"}')

ValidationError: 1 validation error for ExtractedInfo
email
  Field required [type=missing, input_value={'name': 'A'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

In [26]:
ExtractedInfo.model_validate_json('{"name": "A", "email": 5}')

ValidationError: 1 validation error for ExtractedInfo
email
  Input should be a valid string [type=string_type, input_value=5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type

In [27]:
from pydantic import Field

In [28]:
class ExtractedInfo(BaseModel):
    name: str
    email: str = Field(min_length=1)
    phone: str | None = None

In [29]:
ExtractedInfo.model_validate_json('{"name": "A", "email": ""}')

ValidationError: 1 validation error for ExtractedInfo
email
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short

In [30]:
class ExtractedInfo(BaseModel):
    name: str
    email: str | None = None
    phone: str | None = None

In [31]:
for i in range(5):
    response = completion(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": "My name is John Smith"}],
        response_format=ExtractedInfo
    )
    info = ExtractedInfo.model_validate_json(response.choices[0].message.content)
    print(i, info.email)

0 None
1 None
2 None
3 None
4 None


In [32]:
for i in range(5):
    response = completion(
        model="gpt-5-mini",
        messages=[{"role": "system", "content": "Only extract values stated in the text. If a field isn't mentioned, use null. Never guess."},{"role": "user", "content": "My name is John Smith"}],
        response_format=ExtractedInfo
    )
    info = ExtractedInfo.model_validate_json(response.choices[0].message.content)
    print(i, info.email)

0 None
1 None
2 None
3 None
4 None
